In [1]:
import pandas as pd 
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

In [2]:
train = pd.read_csv("train_clean.csv")

train.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV,Total_Charging_Station
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,1,0,1,0,10
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,1,0,1,0,4
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,0,1,1,1,23
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,1,0,1,0,15
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,1,0,1,0,5


In [3]:
train.info()

categorical_cols = ['Gender', 'City_Type', 'Current_Car_Type']

# Ubah tipe data kolom tersebut menjadi category
for col in categorical_cols:
    train[col] = train[col].astype('category')

<class 'pandas.DataFrame'>
RangeIndex: 668665 entries, 0 to 668664
Data columns (total 16 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           668665 non-null  int64  
 1   Age                          668665 non-null  int64  
 2   Annual_Income_USD            668665 non-null  float64
 3   Daily_Commute_km             668665 non-null  float64
 4   Number_of_Cars_Owned         668665 non-null  int64  
 5   Charging_Stations_Near_Home  668665 non-null  int64  
 6   Charging_Stations_Near_Work  668665 non-null  int64  
 7   Environmental_Concern_Level  668665 non-null  float64
 8   Gender                       668665 non-null  str    
 9   City_Type                    668665 non-null  str    
 10  Current_Car_Type             668665 non-null  str    
 11  Home_Charging_Possible       668665 non-null  int64  
 12  Subsidy_Available            668665 non-null  int64  
 13  Range_Anxi

In [4]:
cv = StratifiedKFold(
	n_splits=3,
	shuffle=True,
	random_state=42
)

params = {
	'learning_rate': 0.11890912626024237, 
 	'num_leaves': 35, 
  'max_depth': 5, 
  'min_child_samples': 171, 
  'subsample': 0.8227983632960099, 
  'colsample_bytree': 0.6102452703257967,
  'class_weight': 'balanced',
  'random_state': 42
}

In [5]:
X = train.drop(columns=['id', 'Will_Buy_EV'])
y = train['Will_Buy_EV']

X_train, X_test, y_train, y_test = train_test_split(
	X,
	y,
	test_size=0.2,
	random_state=42,
	stratify=y
)

model = LGBMClassifier(**params)
roc_score = []

for train_idx, val_idx in cv.split(X_train, y_train):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # Training model
    model.fit(X_tr, y_tr)
    
    # Prediksi probabilitas untuk ROC-AUC
    y_pred_proba = model.predict_proba(X_va)[:, 1]
    
    # Hitung skor ROC-AUC per fold
    score = roc_auc_score(y_va, y_pred_proba)
    roc_score.append(score)

print(f"Rata-rata ROC-AUC model (CV): {np.mean(roc_score):.9f}")

[LightGBM] [Info] Number of positive: 62282, number of negative: 294339
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.028230 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 656
[LightGBM] [Info] Number of data points in the train set: 356621, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [6]:
trainf1 = train.copy()
trainf1['income_per_commute'] = train['Annual_Income_USD'] / (train['Daily_Commute_km'] + 1)

X = trainf1.drop(columns=['id', 'Will_Buy_EV'])
y = trainf1['Will_Buy_EV']

X_train, X_test, y_train, y_test = train_test_split(
	X,
	y,
	test_size=0.2,
	random_state=42,
	stratify=y
)

modelf1 = LGBMClassifier(**params)
roc_score = []

for train_idx, val_idx in cv.split(X_train, y_train):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # Training model
    modelf1.fit(X_tr, y_tr)
    
    # Prediksi probabilitas untuk ROC-AUC
    y_pred_proba = modelf1.predict_proba(X_va)[:, 1]
    
    # Hitung skor ROC-AUC per fold
    score = roc_auc_score(y_va, y_pred_proba)
    roc_score.append(score)

print(f"Rata-rata ROC-AUC modelf1 (CV): {np.mean(roc_score):.9f}")

[LightGBM] [Info] Number of positive: 62282, number of negative: 294339
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.056575 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 911
[LightGBM] [Info] Number of data points in the train set: 356621, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [7]:
trainf2 = train.copy()
trainf2['incomexCommute'] = train['Annual_Income_USD'] * train['Daily_Commute_km']

X = trainf2.drop(columns=['id', 'Will_Buy_EV'])
y = trainf2['Will_Buy_EV']

X_train, X_test, y_train, y_test = train_test_split(
	X,
	y,
	test_size=0.2,
	random_state=42,
	stratify=y
)

modelf2 = LGBMClassifier(**params)
roc_score = []

for train_idx, val_idx in cv.split(X_train, y_train):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # Training model
    modelf2.fit(X_tr, y_tr)
    
    # Prediksi probabilitas untuk ROC-AUC
    y_pred_proba = modelf2.predict_proba(X_va)[:, 1]
    
    # Hitung skor ROC-AUC per fold
    score = roc_auc_score(y_va, y_pred_proba)
    roc_score.append(score)

print(f"Rata-rata ROC-AUC modelf2 (CV): {np.mean(roc_score):.9f}")

[LightGBM] [Info] Number of positive: 62282, number of negative: 294339
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.243112 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 911
[LightGBM] [Info] Number of data points in the train set: 356621, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [8]:
trainf3 = train.copy()
trainf3['IncomePerCar'] = train['Annual_Income_USD'] / (train['Number_of_Cars_Owned'] + 1)

X = trainf3.drop(columns=['id', 'Will_Buy_EV'])
y = trainf3['Will_Buy_EV']

X_train, X_test, y_train, y_test = train_test_split(
	X,
	y,
	test_size=0.2,
	random_state=42,
	stratify=y
)

modelf3 = LGBMClassifier(**params)
roc_score = []

for train_idx, val_idx in cv.split(X_train, y_train):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # Training model
    modelf3.fit(X_tr, y_tr)
    
    # Prediksi probabilitas untuk ROC-AUC
    y_pred_proba = modelf3.predict_proba(X_va)[:, 1]
    
    # Hitung skor ROC-AUC per fold
    score = roc_auc_score(y_va, y_pred_proba)
    roc_score.append(score)

print(f"Rata-rata ROC-AUC modelf3 (CV): {np.mean(roc_score):.9f}")

[LightGBM] [Info] Number of positive: 62282, number of negative: 294339
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.048869 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 911
[LightGBM] [Info] Number of data points in the train set: 356621, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [9]:
trainf4 = train.copy()
trainf4['Charging_Income_Interaction'] = train['Total_Charging_Station'] * train['Annual_Income_USD']

X = trainf4.drop(columns=['id', 'Will_Buy_EV'])
y = trainf4['Will_Buy_EV']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

modelf4 = LGBMClassifier(**params)
roc_score = []

for train_idx, val_idx in cv.split(X_train, y_train):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # Training model
    modelf4.fit(X_tr, y_tr)
    
    # Prediksi probabilitas untuk ROC-AUC
    y_pred_proba = modelf4.predict_proba(X_va)[:, 1]
    
    # Hitung skor ROC-AUC per fold
    score = roc_auc_score(y_va, y_pred_proba)
    roc_score.append(score)

print(f"Rata-rata ROC-AUC modelf4 (CV): {np.mean(roc_score):.9f}")

[LightGBM] [Info] Number of positive: 62282, number of negative: 294339
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.213795 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 911
[LightGBM] [Info] Number of data points in the train set: 356621, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [10]:
trainf5 = train.copy()
trainf5['Charging_Range_Interaction'] = train['Total_Charging_Station'] * train['Range_Anxiety_Level']

X = trainf5.drop(columns=['id', 'Will_Buy_EV'])
y = trainf5['Will_Buy_EV']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

modelf5 = LGBMClassifier(**params)
roc_score = []

for train_idx, val_idx in cv.split(X_train, y_train):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # Training model
    modelf5.fit(X_tr, y_tr)
    
    # Prediksi probabilitas untuk ROC-AUC
    y_pred_proba = modelf5.predict_proba(X_va)[:, 1]
    
    # Hitung skor ROC-AUC per fold
    score = roc_auc_score(y_va, y_pred_proba)
    roc_score.append(score)

print(f"Rata-rata ROC-AUC modelf5 (CV): {np.mean(roc_score):.9f}")

[LightGBM] [Info] Number of positive: 62282, number of negative: 294339
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.048432 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 714
[LightGBM] [Info] Number of data points in the train set: 356621, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

Annual_Income_USD/Daily_Commute_km + 1
Annual_Income_USD x Daily_Commute_km
Annual_Income_USD/Number_of_Cars_Owned + 1
Annual_Income_USD/Charging_Stations_Near_Work
Annual_Income_USD/Environmental_Concern_Level

Home_Charging_Possible x Total_Charging_Station

In [11]:
train.groupby("Current_Car_Type")["Will_Buy_EV"].mean()

Current_Car_Type
Hatchback    0.174299
SUV          0.180953
Sedan        0.171967
Truck        0.156413
Name: Will_Buy_EV, dtype: float64

In [12]:
train.groupby("City_Type")["Will_Buy_EV"].mean()

City_Type
Rural       0.193389
Suburban    0.180936
Urban       0.161058
Name: Will_Buy_EV, dtype: float64

In [13]:
train.groupby("Gender")["Will_Buy_EV"].mean()

Gender
Female    0.177641
Male      0.172253
Other     0.173732
Name: Will_Buy_EV, dtype: float64